# BGLC-KG: Personalized Patient Inference Phase (Notebook 3)

**Objective**: This notebook translates the generalized lung cancer knowledge graph (from Notebook 2) into a **Precision Medicine / Personalized Inference Engine**. 
It takes an individual patient's genomic profile (mutated genes) and uses the pre-trained HeteroGraphSAGE embeddings to rank drugs specifically tailored to that patient's unique biological network.

> **Future Update Placeholder**: Currently, this notebook runs on a simulated Bangladeshi patient profile (Base System). In 3-4 months, this will be updated to ingest primary VCF/CSV dataset files from real hospital cohorts for ultimate clinical validation.

In [ ]:
# 1. Environment Setup & Imports
import os
import torch
import numpy as np
import pandas as pd
import networkx as nx
import seaborn as sns
import matplotlib.pyplot as plt
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, to_hetero

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

BASE_DIR = os.getcwd() if 'kaggle' not in os.getcwd() else '/kaggle/working'
MODEL_DIR = os.path.join(BASE_DIR, 'models')
GRAPH_DIR = os.path.join(BASE_DIR, 'graph')
VIS_DIR = os.path.join(BASE_DIR, 'visualizations')
os.makedirs(VIS_DIR, exist_ok=True)

In [ ]:
# 2. Load Pre-trained Knowledge Graph
graph_path = os.path.join(GRAPH_DIR, 'BGLC-KG-split.pt')
if not os.path.exists(graph_path):
    raise FileNotFoundError(f"Graph file not found at {graph_path}. Please run Notebook 1 first.")
    
data = torch.load(graph_path).to(device)
print("Knowledge Graph Loaded Successfully.")
print(f"Node Types: {data.node_types}")

In [ ]:
# 3. Redefine HeteroGraphSAGE Architecture to Load Weights
class BaseEncoder(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv((-1, -1), hidden_channels)
        self.conv2 = SAGEConv((-1, -1), out_channels)
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv2(x, edge_index)

class EdgeDecoder(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.lin1 = torch.nn.Linear(2 * hidden_channels, hidden_channels)
        self.lin2 = torch.nn.Linear(hidden_channels, 1)
    def forward(self, z_dict, edge_label_index, target_edge_type):
        row, col = edge_label_index
        z = torch.cat([z_dict[target_edge_type[0]][row], z_dict[target_edge_type[2]][col]], dim=-1)
        z = self.lin1(z).relu()
        return self.lin2(z).view(-1)

class HeteroLinkPredictionModel(torch.nn.Module):
    def __init__(self, data, hidden_channels):
        super().__init__()
        self.encoder = to_hetero(BaseEncoder(hidden_channels, hidden_channels), data.metadata(), aggr='sum')
        self.decoder = EdgeDecoder(hidden_channels)
    def forward(self, x_dict, edge_index_dict, edge_label_index):
        z_dict = self.encoder(x_dict, edge_index_dict)
        return self.decoder(z_dict, edge_label_index, ('drug', 'indicated_for', 'disease'))

hidden_dim = 64
model = HeteroLinkPredictionModel(data, hidden_dim).to(device)
model_path = os.path.join(MODEL_DIR, 'best_heterosage_model.pth')

try:
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    print("Pre-trained HeteroGraphSAGE Model Loaded Successfully!")
except Exception as e:
    print(f"Warning: Could not load model weights ({e}). Make sure Notebook 2 is run first.")

In [ ]:
# 4. Define the Dummy Patient Profile (The BASE for Future Validation)
# -------------------------------------------------------------------
# FUTURE UPDATE: In 3-4 months, replace this mock dictionary with a CSV loader
# e.g., df_patient = pd.read_csv('primary_clinical_dataset.csv')
# -------------------------------------------------------------------

print("\n--- Initializing Personalized Patient Profile ---")
patient_profile = {
    "patient_id": "PT_BD_001",
    "demographics": "Bangladeshi, Male, 55",
    "cancer_type": "NSCLC (Non-Small Cell Lung Cancer)",
    # These are the specific target nodes in the graph the patient has mutations in
    # We assume gene indices 0 (EGFR) and 5 (KRAS) for this dummy baseline.
    "mutated_gene_indices": [0, 5], 
    "vaf_scores": [0.45, 0.30] # Variant Allele Frequencies (Future weighting feature)
}

print(f"Patient ID: {patient_profile['patient_id']}")
print(f"Mutated Gene Graph Indices: {patient_profile['mutated_gene_indices']}")

In [ ]:
# 5. Personalized Inference Engine (Extracting Patient-Specific Latent Vectors)
# We extract the GNN embeddings. A drug is scored based on:
# 1. Its global affinity to the disease (Lung Cancer)
# 2. Its personalized affinity to the patient's specific mutated genes in the latent space.

with torch.no_grad():
    # Extract all node embeddings from the trained GNN
    z_dict = model.encoder(data.x_dict, data.edge_index_dict)
    drug_embeddings = z_dict['drug']
    disease_embeddings = z_dict['disease']
    gene_embeddings = z_dict['gene']
    
    # 1. Global Disease Affinity
    target_disease_idx = 0
    num_drugs = data['drug'].num_nodes
    candidate_edge_label_index = torch.stack([
        torch.arange(num_drugs),
        torch.full((num_drugs,), target_disease_idx, dtype=torch.long)
    ], dim=0).to(device)
    
    base_logits = model.decoder(z_dict, candidate_edge_label_index, ('drug', 'indicated_for', 'disease'))
    global_probs = torch.sigmoid(base_logits).cpu().numpy()
    
    # 2. Personalized Gene Affinity (Cosine Similarity in Latent Space)
    # Average the embeddings of the patient's mutated genes
    mutated_gene_vectors = gene_embeddings[patient_profile['mutated_gene_indices']]
    patient_genetic_vector = mutated_gene_vectors.mean(dim=0).unsqueeze(0) # [1, hidden_dim]
    
    # Compute cosine similarity between all drugs and the patient's genetic vector
    from torch.nn.functional import cosine_similarity
    drug_personal_affinity = cosine_similarity(drug_embeddings, patient_genetic_vector).cpu().numpy()
    
    # 3. Final Personalized Score = 0.7 * Global Evidence + 0.3 * Patient Specific Affinity
    personalized_scores = (0.7 * global_probs) + (0.3 * drug_personal_affinity)
    
    # Exclude already known drugs for this disease to find purely novel personalized recommendations
    known_edges = data[('drug', 'indicated_for', 'disease')].edge_index
    known_mask = known_edges[1] == target_disease_idx
    known_drugs = set(known_edges[0][known_mask].numpy())
    
    results = []
    for d_idx in range(num_drugs):
        if d_idx not in known_drugs:
            results.append({
                'Drug_Node_ID': d_idx,
                'Global_Probability': global_probs[d_idx],
                'Patient_Affinity': drug_personal_affinity[d_idx],
                'Final_Personalized_Score': personalized_scores[d_idx]
            })
            
    df_patient_drugs = pd.DataFrame(results).sort_values(by='Final_Personalized_Score', ascending=False)
    
    print(f"\nTop 10 Personalized Drug Candidates for {patient_profile['patient_id']}:")
    print(df_patient_drugs.head(10).to_string(index=False))
    
    csv_path = os.path.join(MODEL_DIR, f"personalized_preds_{patient_profile['patient_id']}.csv")
    df_patient_drugs.to_csv(csv_path, index=False)
    print(f"\nPersonalized Predictions saved to {csv_path}")

In [ ]:
# 6. XAI: Visualizing the Personalized Patient Subgraph
print("\n--- Generating Personalized XAI Subgraph ---")
top_drug_idx = int(df_patient_drugs.iloc[0]['Drug_Node_ID'])
score = df_patient_drugs.iloc[0]['Final_Personalized_Score']

G_patient = nx.Graph()
# Core nodes
G_patient.add_node(patient_profile['patient_id'], color='gold', size=1200)
G_patient.add_node(f"Disease_NSCLC", color='green', size=800)
G_patient.add_node(f"Drug_{top_drug_idx}", color='red', size=1000)

# Connect Patient to Disease
G_patient.add_edge(patient_profile['patient_id'], f"Disease_NSCLC", label="diagnosed_with")

# Connect Patient to their specific mutated genes
for g_idx in patient_profile['mutated_gene_indices']:
    G_patient.add_node(f"Gene_{g_idx}", color='lightblue', size=600)
    G_patient.add_edge(patient_profile['patient_id'], f"Gene_{g_idx}", label="has_mutation")
    
# Connect Drug to Disease (The Prediction)
G_patient.add_edge(f"Drug_{top_drug_idx}", f"Disease_NSCLC", label=f"Personalized Pred ({score:.2f})")

# Find if the predicted drug targets the patient's specific genes
drug_targets_gene = data[('drug', 'targets', 'gene')].edge_index
drug_mask = drug_targets_gene[0] == top_drug_idx
targeted_genes = drug_targets_gene[1][drug_mask].cpu().numpy()

for g_idx in targeted_genes:
    if g_idx in patient_profile['mutated_gene_indices']:
        G_patient.add_edge(f"Drug_{top_drug_idx}", f"Gene_{g_idx}", label="DIRECT TARGET")
    else:
        # Drug targets other genes too, let's show a couple for context
        if len(list(G_patient.nodes)) < 10:
            G_patient.add_node(f"Gene_{g_idx}", color='lightgrey', size=400)
            G_patient.add_edge(f"Drug_{top_drug_idx}", f"Gene_{g_idx}", label="targets")

plt.figure(figsize=(12, 9))
pos = nx.spring_layout(G_patient, seed=42, k=0.8)
colors = [nx.get_node_attributes(G_patient, 'color').get(n, 'grey') for n in G_patient.nodes()]
sizes = [nx.get_node_attributes(G_patient, 'size').get(n, 500) for n in G_patient.nodes()]
nx.draw(G_patient, pos, with_labels=True, node_color=colors, node_size=sizes, font_size=10, font_weight='bold', edge_color='gray')
edge_labels = nx.get_edge_attributes(G_patient, 'label')
nx.draw_networkx_edge_labels(G_patient, pos, edge_labels=edge_labels, font_size=9, font_color='darkred')
plt.title(f"XAI: Personalized Medicine Graph for {patient_profile['patient_id']}", fontweight='bold', fontsize=14)
plt.savefig(os.path.join(VIS_DIR, f"personalized_xai_{patient_profile['patient_id']}.png"), dpi=300)
plt.show()

print("\nPersonalized Inference Complete! Notebook 3 Ready.")